This notebook performs **post-training evaluation only**. No model parameters are updated here.  
We load the trained **ColdFusionX** checkpoint produced in Notebook 08 along with the saved configuration files.
The goal is to ensure a **reproducible evaluation environment** before computing overall and cold/warm metrics on the held-out validation/test sets.

In [30]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
import os
import json
import torch
import random
import pandas as pd
import numpy as np

In [32]:
# Base paths
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
checkpoints_dir = os.path.join(base_dir, "checkpoints")   # where final_model.pt is saved (Notebook 08)
results_dir = os.path.join(base_dir, "eval_results")      # output folder for Notebook 09

os.makedirs(results_dir, exist_ok=True)

In [33]:
print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("checkpoints_dir:", checkpoints_dir)
print("results_dir:", results_dir)

print("\nExists? ")
print("base_dir exists?        ", os.path.exists(base_dir))
print("preprocessed_dir exists?", os.path.exists(preprocessed_dir))
print("checkpoints_dir exists? ", os.path.exists(checkpoints_dir))

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints
results_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results

Exists? 
base_dir exists?         True
preprocessed_dir exists? True
checkpoints_dir exists?  True


In [34]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [35]:
set_seed(42)

# Determinism (can slightly reduce performance)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed set to 42. Deterministic flags enabled.")

Seed set to 42. Deterministic flags enabled.


In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("GPU mem (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

Device: cpu


In [37]:
# Configs (from 06/07)
fusion_config_path = os.path.join(checkpoints_dir, "fusion_config.json")
meta_config_path   = os.path.join(checkpoints_dir, "meta_config.json")

# Training artifacts (from 08)
final_model_path   = os.path.join(checkpoints_dir, "coldfusionx_model_epoch3.pt")
training_config_path = os.path.join(checkpoints_dir, "training_meta_epoch3.json")

# Splits (from 07)
splits_dir = os.path.join(preprocessed_dir, "splits")
train_idx_path = os.path.join(splits_dir, "train_idx.npy")
val_idx_path   = os.path.join(splits_dir, "val_idx.npy")
test_idx_path  = os.path.join(splits_dir, "test_idx.npy")

paths = {
    "fusion_config": fusion_config_path,
    "meta_config": meta_config_path,
    "final_model": final_model_path,
    "training_config": training_config_path,
    "train_idx": train_idx_path,
    "val_idx": val_idx_path,
    "test_idx": test_idx_path,
}

In [38]:
print("Artifact Paths")
for k, p in paths.items():
    print(f"{k:>16}: {p} | exists? {os.path.exists(p)}")

Artifact Paths
   fusion_config: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/fusion_config.json | exists? True
     meta_config: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/meta_config.json | exists? True
     final_model: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/coldfusionx_model_epoch3.pt | exists? True
 training_config: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/training_meta_epoch3.json | exists? True
       train_idx: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/splits/train_idx.npy | exists? True
         val_idx: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/splits/val_idx.npy | exists? True
        test_idx: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_prepr

In [39]:
# Load JSON configs
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

with open(training_config_path, "r") as f:
    training_config = json.load(f)

In [41]:
# Load split indices
train_idx = np.load(train_idx_path)
val_idx   = np.load(val_idx_path)
test_idx  = np.load(test_idx_path)

print("Loaded Splits")
print("train:", len(train_idx), "val:", len(val_idx), "test:", len(test_idx))

Loaded Splits
train: 2984263 val: 373033 test: 373033


#### Load Row Aligned Evaluation Artifacts
Here, we load all **interaction-level** tensors that must share the same row alignment:
- `df_id_encoded.pkl` (user/item/scene indices per interaction)
- `text_embeddings.npy` (memmap, interaction-level)
- `time_features_final.pt` (interaction-level)
- `label` vector (interaction-level)

We also load **item-level** image embeddings (`image_embeddings_vit_base_aligned.pt`) which must match the item vocabulary size. We then run strict assertions to confirm that all interaction-level artifacts have the expected length **N = 3,730,329** and that item-level embeddings match `(num_items, 768)`.

In [42]:
df_id_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")
text_path  = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_path  = os.path.join(preprocessed_dir, "time_features_final.pt")

# Image artifacts (item-level)
image_emb_path  = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

# Labels
df_processed_path = os.path.join(preprocessed_dir, "df_processed.pkl")

In [43]:
print("Artifact Exists?")
for k, p in {
    "df_id_encoded.pkl": df_id_path,
    "text_embeddings.npy": text_path,
    "time_features_final.pt": time_path,
    "image_embeddings_vit_base_aligned.pt": image_emb_path,
    "image_missing_mask.pt": image_mask_path,
    "df_processed.pkl": df_processed_path,
}.items():
    print(f"{k:>32}: {p} | exists? {os.path.exists(p)}")

Artifact Exists?
               df_id_encoded.pkl: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features/df_id_encoded.pkl | exists? True
             text_embeddings.npy: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy | exists? True
          time_features_final.pt: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt | exists? True
image_embeddings_vit_base_aligned.pt: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt | exists? True
           image_missing_mask.pt: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_missing_mask.pt | exists? True
                df_processed.pkl: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/df_processed.pkl | exis

In [46]:
# Load df_id (interaction-level indices)
df_id = pd.read_pickle(df_id_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", list(df_id.columns))

# Basic column checks
required_cols = ["user_id_idx", "item_id_idx", "scene_idx"]
missing_cols = [c for c in required_cols if c not in df_id.columns]
assert not missing_cols, f"df_id is missing required columns: {missing_cols}"

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']


In [48]:
# Load text embeddings as memmap (interaction-level)
text_mem = np.load(text_path, mmap_mode="r")
print("Loaded text_embeddings.npy (memmap):", text_mem.shape, "| dtype:", text_mem.dtype)

Loaded text_embeddings.npy (memmap): (3730329, 1536) | dtype: float32


In [49]:
# Load time features (interaction-level)
time_features = torch.load(time_path, map_location="cpu")
print("Loaded time_features_final.pt:", tuple(time_features.shape), "| dtype:", time_features.dtype)

Loaded time_features_final.pt: (3730329, 5) | dtype: torch.float32


In [51]:
# Load image embeddings + missing mask (item-level)
image_embeddings = torch.load(image_emb_path, map_location="cpu")
image_missing_mask = torch.load(image_mask_path, map_location="cpu")

print("Loaded image_embeddings:", tuple(image_embeddings.shape), "| dtype:", image_embeddings.dtype)
print("Loaded image_missing_mask:", tuple(image_missing_mask.shape),
      "| missing count:", int(image_missing_mask.sum().item()))

Loaded image_embeddings: (59142, 768) | dtype: torch.float32
Loaded image_missing_mask: (59142,) | missing count: 2


In [52]:
# Load labels (interaction-level)
df_processed = pd.read_pickle(df_processed_path)

In [55]:
label_col = "label"
assert label_col in df_processed.columns, f"'{label_col}' not found in df_processed.pkl columns: {df_processed.columns}"

labels = df_processed[label_col].to_numpy(dtype=np.float32)
pos_rate = float(labels.mean())
print("Loaded labels:", labels.shape, "| positives rate:", pos_rate)

Loaded labels: (3730329,) | positives rate: 0.33633333444595337


In [56]:
N = len(df_id)
print("Alignment Checks")
print("N (df_id rows):", N)

assert text_mem.shape[0] == N, f"text_mem rows ({text_mem.shape[0]}) != df_id rows ({N})"
assert time_features.shape[0] == N, f"time_features rows ({time_features.shape[0]}) != df_id rows ({N})"
assert labels.shape[0] == N, f"labels length ({labels.shape[0]}) != df_id rows ({N})"

Alignment Checks
N (df_id rows): 3730329


In [57]:
# Item-level alignment checks
num_items_expected = int(fusion_config["num_items"]) if "fusion_config" in globals() else image_embeddings.shape[0]
assert image_embeddings.shape[0] == num_items_expected, (
    f"image_embeddings rows ({image_embeddings.shape[0]}) != expected num_items ({num_items_expected})"
)
assert image_embeddings.shape[1] == 768, f"image_embeddings dim ({image_embeddings.shape[1]}) != 768"

assert image_missing_mask.shape[0] == image_embeddings.shape[0], (
    "image_missing_mask length must match image_embeddings rows"
)

In [59]:
!git config --global user.email "sharmfernando33@gmail.com"
!git config --global user.name "sheebaboy33"

Refresh index: 100% (14/14), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/06_fusion_module.ipynb
	modified:   notebooks/08_predictor_training.ipynb
	modified:   notebooks/09_evaluation.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
